# 03 - Data Cleaning

Day 1, step 3. Notebook 01 found the problems; this fixes them and writes a clean
copy of every dataset to `data/processed/`. The raw files are never overwritten.

The transformations live in `pipelines/clean_data.py` so the API and the batch jobs
clean data identically. This notebook runs that pipeline and inspects what it did.

In [1]:
import pandas as pd

from app.services.skill_normalizer import normalization_audit, normalize_skill
from app.utils.config import PROCESSED_FILES, RAW_FILES
from pipelines import clean_data

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

raw_emp = pd.read_csv(RAW_FILES["attrition"], encoding="utf-8-sig")
raw_emp.columns = [c.strip() for c in raw_emp.columns]
print("raw attrition shape:", raw_emp.shape)

raw attrition shape: (1470, 35)


## Run the cleaning pipeline

In [2]:
cleaned = clean_data.main()

2026-09-02 14:05:22 | INFO    | hr_ai.clean | attrition cleaned: (1470, 35) -> (1470, 33) (dropped constant columns ['EmployeeCount', 'Over18', 'StandardHours'])


2026-09-02 14:05:22 | INFO    | hr_ai.clean | engagement cleaned: (1470, 12)


2026-09-02 14:05:22 | INFO    | hr_ai.clean | occupation master cleaned: (1016, 3)


2026-09-02 14:05:22 | INFO    | hr_ai.clean | essential skills cleaned: (9100, 4)


2026-09-02 14:05:23 | INFO    | hr_ai.clean | software skills cleaned: (31779, 6)


2026-09-02 14:05:23 | INFO    | hr_ai.clean | role requirements cleaned: (162, 7)


2026-09-02 14:05:23 | INFO    | hr_ai.clean | employee skills cleaned: (12388, 5)


employee_attrition_processed.csv                1,470 rows x 33 cols

engagement_processed.csv                        1,470 rows x 12 cols
occupation_master.csv                           1,016 rows x  3 cols
essential_skills_processed.csv                  9,100 rows x  4 cols


software_skills_processed.csv                  31,779 rows x  6 cols
role_skill_requirements_processed.csv             162 rows x  7 cols
employee_skills_processed.csv                  12,388 rows x  5 cols


## What changed in the employee table

In [3]:
emp = cleaned["attrition"]
removed = sorted(set(raw_emp.columns) - set(emp.columns))
added = sorted(set(emp.columns) - set(raw_emp.columns))

print(f"columns: {raw_emp.shape[1]} -> {emp.shape[1]}")
print(f"  removed: {removed}")
print(f"  added  : {added}")
print(f"rows   : {len(raw_emp)} -> {len(emp)}")

columns: 35 -> 33
  removed: ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
  added  : ['AttritionFlag', 'EmployeeID']
rows   : 1470 -> 1470


`EmployeeNumber` is renamed to `EmployeeID`, so it appears in both lists. The three
genuinely dropped columns are the zero-variance ones found in notebook 01, and
`AttritionFlag` is the 0/1 encoding of the target for modelling - the original
`Attrition` Yes/No column is kept for reporting.

In [4]:
print("Attrition vs AttritionFlag agree:",
      bool((emp["AttritionFlag"] == (emp["Attrition"] == "Yes").astype(int)).all()))
print(emp[["EmployeeID", "Attrition", "AttritionFlag"]].head())

Attrition vs AttritionFlag agree: True
   EmployeeID Attrition  AttritionFlag
0           1       Yes              1
1           2        No              0
2           4       Yes              1
3           5        No              0
4           7        No              0


## Outliers: reported, not removed

In [5]:
outliers = clean_data.report_outliers(
    emp, ["MonthlyIncome", "TotalWorkingYears", "YearsAtCompany",
          "YearsSinceLastPromotion", "DistanceFromHome", "Age"])
outliers

,column,q1,q3,lower_fence,upper_fence,n_outliers,pct
0,MonthlyIncome,2911.0,8379.0,-5291.0,16581.0,114,7.76
1,TotalWorkingYears,6.0,15.0,-7.5,28.5,63,4.29
2,YearsAtCompany,3.0,9.0,-6.0,18.0,104,7.07
3,YearsSinceLastPromotion,0.0,3.0,-4.5,7.5,107,7.28
4,DistanceFromHome,2.0,14.0,-16.0,32.0,0,0.00
5,Age,30.0,43.0,10.5,62.5,0,0.00


`MonthlyIncome` flags around 8% of rows as IQR outliers. Those are directors and
managers, not corrupted values - the column is genuinely right-skewed, as notebook 01
showed. Clipping or dropping them would delete the most senior slice of the
workforce, which is precisely the population where a wrong attrition call is most
expensive.

So: flagged and documented, values untouched. Tree models handle skew natively, and
the linear baseline gets a scaler instead.

In [6]:
top_earners = emp.nlargest(5, "MonthlyIncome")[
    ["EmployeeID", "JobRole", "JobLevel", "TotalWorkingYears", "MonthlyIncome"]]
print("The 'outliers' in MonthlyIncome:")
print(top_earners.to_string(index=False))

The 'outliers' in MonthlyIncome:
 EmployeeID           JobRole  JobLevel  TotalWorkingYears  MonthlyIncome
        259           Manager         5                 34          19999
       1035 Research Director         5                 21          19973
       1191           Manager         5                 28          19943
        226           Manager         5                 21          19926
        787           Manager         5                 24          19859


All five are Managers or Research Directors with long careers. Real people, real
salaries.

## Skill name canonicalisation

The problem from the build notes: `AWS`, `Amazon Web Services` and `AWS Cloud` are
one skill. Left split, the gap engine counts one shortage three times and recommends
training an employee already has.

In [7]:
examples = [
    "AWS", "Amazon Web Services", "AWS Cloud", "Amazon Web Services AWS software",
    "SAP software", "Salesforce software", "The MathWorks MATLAB",
    "Micosoft SQL Server Analysis Services SSAS", "Microsoft Office software",
    "  excel  ", "Applicant tracking software", "Electronic medical record EMR software",
]
print(f"{'raw':45s} -> canonical")
for e in examples:
    print(f"{e!r:45s} -> {normalize_skill(e)!r}")

raw                                           -> canonical
'AWS'                                         -> 'Amazon Web Services (AWS)'
'Amazon Web Services'                         -> 'Amazon Web Services (AWS)'
'AWS Cloud'                                   -> 'Amazon Web Services (AWS)'
'Amazon Web Services AWS software'            -> 'Amazon Web Services (AWS)'
'SAP software'                                -> 'SAP'
'Salesforce software'                         -> 'Salesforce'
'The MathWorks MATLAB'                        -> 'MATLAB'
'Micosoft SQL Server Analysis Services SSAS'  -> 'Microsoft SQL Server Analysis Services SSAS'
'Microsoft Office software'                   -> 'Microsoft Office'
'  excel  '                                   -> 'Microsoft Excel'
'Applicant tracking software'                 -> 'Applicant Tracking System (ATS)'
'Electronic medical record EMR software'      -> 'Electronic Medical Record (EMR)'


### What it actually merged in this data

Worth checking honestly rather than assuming the normaliser earned its keep.

In [8]:
raw_soft = pd.read_csv(RAW_FILES["software_skills"])
audit = normalization_audit(raw_soft["Workplace Example"].dropna().unique())

print(f"canonical names that absorbed more than one raw spelling: {len(audit)}")
for canon, raws in list(audit.items())[:10]:
    print(f"  {canon!r}")
    for r in raws:
        print(f"      <- {r!r}")

canonical names that absorbed more than one raw spelling: 24
  'Bioconductor'
      <- 'Bioconductor'
      <- 'Bioconductor software'
  'Business intelligence'
      <- 'Business intelligence software'
      <- 'Business intelligence system software'
  'Case management'
      <- 'Case management software'
      <- 'Case management system software'
  'Client information database'
      <- 'Client information database software'
      <- 'Client information database systems'
  'Contact management'
      <- 'Contact management software'
      <- 'Contact management systems'
  'Data acquisition'
      <- 'Data acquisition software'
      <- 'Data acquisition systems'
  'Database management'
      <- 'Database management software'
      <- 'Database management system software'
      <- 'Database management systems'
  'Digital imaging'
      <- 'Digital imaging software'
      <- 'Digital imaging system software'
  'ESRI ArcGIS'
      <- 'ESRI ArcGIS software'
      <- 'Esri ArcGIS'
  'Elect

In [9]:
soft_clean = cleaned["software_skills"]
print(f"software_skills rows: {len(raw_soft):,} -> {len(soft_clean):,} "
      f"({len(raw_soft) - len(soft_clean)} merged as duplicate (role, skill) pairs)")

req = cleaned["role_requirements"]
print(f"distinct required skills across all roles: {req['SkillName'].nunique()}")

software_skills rows: 31,821 -> 31,779 (42 merged as duplicate (role, skill) pairs)
distinct required skills across all roles: 52


Within the role-requirement vocabulary specifically, every raw name was already
distinct, so normalisation merged nothing there. It still matters: it is what keeps
the *employee* skills table joinable to requirements, and it is what will absorb the
abbreviations ("AWS", "Excel") that a real HR skills export contains.

## Re-validate after cleaning

Cleaning is itself a transformation that can introduce bugs, so the same rules run
again on the output.

In [10]:
from app.validation.employee_schema import validate_employee_frame
from app.validation.engagement_schema import (
    validate_employee_skills_frame,
    validate_engagement_frame,
)

print(validate_employee_frame(emp))
print(validate_engagement_frame(cleaned["engagement"]))
print(validate_employee_skills_frame(cleaned["employee_skills"],
                                     valid_employee_ids=set(emp["EmployeeID"])))

[PASS] employee_attrition: 1,470 rows, 25 checks, 0 errors, 0 warnings
[PASS] hr_performance_engagement: 1,470 rows, 18 checks, 0 errors, 0 warnings
[PASS] employee_current_skills: 12,388 rows, 6 checks, 0 errors, 0 warnings


## Output

In [11]:
for key in ["attrition", "engagement", "occupation", "essential_skills",
            "software_skills", "role_requirements", "employee_skills"]:
    path = PROCESSED_FILES[key]
    print(f"{path.name:45s} {path.stat().st_size / 1_000:>9,.0f} KB")

employee_attrition_processed.csv                    221 KB
engagement_processed.csv                            124 KB
occupation_master.csv                               268 KB
essential_skills_processed.csv                      597 KB
software_skills_processed.csv                     3,227 KB
role_skill_requirements_processed.csv                15 KB
employee_skills_processed.csv                       541 KB


Every dataset now has a clean copy, keyed consistently on `EmployeeID`, with
canonical skill names. Notebook 04 confirms how they join.